# Deep Models

This notebook explores pretrained image models available in `src.deep_models`.
It demonstrates how to load models on CPU or CUDA, inspect intermediate feature maps,
and visualize a ResNet-18 class activation map (CAM). The final section compares
model architectures used for feature extraction.

The examples use the image at `data/images/000044.jpg`. Change `image_path` below
to inspect another image.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image

from src.deep_models import getModel

image_path = Path("data/images/000007.jpg")
if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

image = Image.open(image_path).convert("RGB")
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")
print(f"Image size: {image.size}")

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off")
plt.title(image_path.name)
plt.show()

## ResNet-18 Intermediate Features and CAM

The feature extractor records outputs from compound blocks during a forward pass.
For ResNet-18, `layer4.1` is the final convolutional block and produces the feature
map used below for class activation visualization.

### Inspect Intermediate Feature Maps

The extractor returns the model logits together with a dictionary keyed by the
captured block name. The spatial resolution decreases through the ResNet stages,
while the channel count increases. These tensors are useful for comparing early
and late visual representations.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchinfo import summary
from torchvision.models import ResNet18_Weights

from src.deep_models import IntermediateFeatureExtractor, getModel

# Keep the pretrained preprocessing that matches the model weights.
resnet, preprocess = getModel("resnet18", device=device, nofc=False)
print(summary(resnet, input_size=(1, 3, 224, 224), depth=2))

# Simple inference with the ResNet-18 model to get class probabilities.
transformed_image = preprocess(image).unsqueeze(0).to(device)
with torch.inference_mode():
    logits = resnet(transformed_image)
    # Convert logits into class probabilities.
    probabilities = torch.softmax(logits, dim=1)

    # Select the five most likely ImageNet classes.
    top_probabilities, top_indices = probabilities[0].topk(5)

    # Retrieve the class names associated with the predicted indices.
    categories = ResNet18_Weights.DEFAULT.meta["categories"]

    # Display the predictions and their confidence scores.
    print("Top predictions:")
    for probability, index in zip(top_probabilities, top_indices):
        print(f"{categories[index.item()]}: {probability.item():.2%}")

In [ ]:
# Extract intermediate feature maps from the ResNet-18 model.
###############################################################################
extractor = IntermediateFeatureExtractor(resnet)
with torch.inference_mode():
    logits, intermediate = extractor(transformed_image)

print("Captured blocks:", list(intermediate))
for layer_name in ("layer2.1", "layer3.1", "layer4.1"):
    feature_map = intermediate[layer_name]
    print(f"{layer_name} feature map: {tuple(feature_map.shape)}")
print(f"Logits shape: {tuple(logits.shape)}")

In [ ]:
# Class Activation Map (CAM) visualization for ResNet-18.
classes = ResNet18_Weights.DEFAULT.meta["categories"]


def resnet_cam(image, extractor, preprocess, target_class, classes, device):
    """Display a normalized ResNet-18 CAM for one ImageNet target class."""
    target_index = classes.index(target_class)
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        _, features = extractor(image_tensor)
        feature_map = features["layer4.1"]
        weights = extractor.model.fc.weight[target_index]
        cam = F.relu((feature_map[0] * weights[:, None, None]).sum(dim=0))
        cam = F.interpolate(
            cam[None, None],
            size=(image.height, image.width),
            mode="bilinear",
            align_corners=False,
        )[0, 0].cpu()
        cam = cam / cam.max().clamp_min(1e-8)

    print(f"Target class: {target_class} ({target_index})")
    print(f"CAM shape: {tuple(cam.shape)}")

    fig, axis = plt.subplots(figsize=(9, 6))
    axis.imshow(image)
    axis.imshow(cam, cmap="jet", alpha=0.5, vmin=0, vmax=1)
    axis.set_title(f"ResNet-18 CAM: {target_class}")
    axis.axis("off")
    plt.show()
    return cam


initial_class = classes[top_indices[0].item()]

target_class = widgets.Combobox(
    options=classes,
    value=initial_class,
    description="Class:",
    ensure_option=True,
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)
output = widgets.Output()


def update_cam(change):
    if change["name"] == "value" and change["new"]:
        with output:
            output.clear_output(wait=True)
            resnet_cam(image, extractor, preprocess, change["new"], classes, device)


target_class.observe(update_cam, names="value")
display(target_class, output)
update_cam({"name": "value", "new": initial_class})

## Compare Feature Backbones

DINOv2 and CLIP provide alternative pretrained representations. These cells load
the models on the selected device and print their architecture summaries. DINOv2
may download weights from Torch Hub the first time it is used.

The summaries below are architectural inspections, not a benchmark. For a fair
embedding comparison, apply each model's returned preprocessing transform and
measure the resulting feature tensor on the same image.

In [ ]:
dinov2, dino_preprocess = getModel("dinov2_vits14", device=device)
print(summary(dinov2, input_size=(1, 3, 224, 224), depth=2))

In [ ]:
clip_model, clip_preprocess = getModel("clip_vit_b_32", device=device)
print(summary(clip_model, depth=2))
print(summary(clip_model.visual, depth=3))
print(summary(clip_model.transformer, depth=2))

## CLIP Zero-Shot Classification

CLIP compares an image embedding with text embeddings in the same representation
space. The labels below are only an example; replace them with any candidate
descriptions relevant to the reference image. The scores are relative to this
candidate list and are not calibrated class probabilities.

In [ ]:
import clip

candidate_labels = [
    "a photo of an aeroplane",
    "a photo of a car",
    "a photo of a horse",
    "a photo of a motorbike",
    "a photo of a person",
    "a photo of a background scene",
]

text_tokens = clip.tokenize(candidate_labels).to(device)
image_tensor = clip_preprocess(image).unsqueeze(0).to(device)

with torch.inference_mode():
    image_features = clip_model.encode_image(image_tensor)
    text_features = clip_model.encode_text(text_tokens)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    scores = (100.0 * image_features @ text_features.T).softmax(dim=-1)[0]

ranking = torch.argsort(scores, descending=True)
print("Zero-shot ranking:")
for rank, label_index in enumerate(ranking, start=1):
    print(f"{rank}. {candidate_labels[label_index]}: {scores[label_index].item():.3f}")